# Four-model comparisons and honest acceptance

Compare four algorithms independently for applicant risk and customer-profile association. Selection, calibration and thresholding use training-only folds. Preserve the original application holdout instead of trying favourable seeds.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
OUT = ROOT / 'artifacts'
assert json.loads((OUT / 'run_manifest.json').read_text())['version'] == 2, 'Run python -m src.pipeline first'

main=json.loads((OUT/'metrics.json').read_text())
customer=json.loads((OUT/'customer_model/metrics.json').read_text())
for name,result in [('Applicant',main),('Customer profile',customer)]:
    print(name,result['champion'])
    display(pd.DataFrame([{k:v for k,v in row.items() if k not in ['parameters','test_at_0_5']} for row in result['comparison']]))

## Validation protocol
The applicant model uses the frozen 80/20 application partition, previously reported in v1, so the continuation test is not fresh external validation. Five-fold training CV selects hyperparameters and algorithm. Training-only sigmoid calibration uses three-fold folds; five-fold calibrated OOF probabilities select a threshold with at least 80% recall. The customer model has one latest observed label per borrower and a customer-disjoint stratified holdout.

In [ ]:
for name,result in [('Applicant',main),('Customer',customer)]:
    print(name,'threshold',result['threshold'],'shared test customers',result['test_customer_overlap'])
    display(pd.DataFrame([{k:v for k,v in result['test_metrics'].items() if k!='confusion_matrix'}]))
    print('Confusion matrix:',result['test_metrics']['confusion_matrix'])
    print('Acceptance:',result['success_targets'])
display(Image(filename=str(OUT/'figures/model_evaluation.png')))

In [ ]:
curve=pd.read_csv(OUT/'threshold_curve.csv')
curve.plot(x='threshold',y=['precision','recall','f1'],figsize=(9,4));plt.axvline(main['threshold'],color='black',linestyle='--');plt.title('Training-only review threshold');plt.show()

## Calendar and borrower sensitivity
The borrower-disjoint audit reveals repeated-customer optimism. The fixed calendar audit trains before January 2024 and tests 2024 requests. Because outcome-event dates and observation ends are absent, this calendar split cannot prove that training labels had matured at the cutoff.

In [ ]:
print('Borrower-disjoint audit:',main['group_holdout_audit'])
print('Calendar audit:',json.loads((OUT/'temporal_audit.json').read_text()))

## AUC investigation
The separate wider retrospective search uses only the frozen development records. Log transforms, credit thresholds and wider regularisation are investigated without the holdout. Its CV estimate is not substituted for a failed held-out target and its snapshots remain unsuitable for historical request-time prediction.

In [ ]:
path=OUT/'auc_investigation/development_cv.json'
if path.exists():
    study=json.loads(path.read_text());print('Test set used:',study['test_set_used']);display(pd.DataFrame(study['comparison']))
print(json.loads((OUT/'acceptance_status.json').read_text()))

## Explanation scales
Every applicant and customer has complete signed SHAP traces. Contributions reconstruct the selected base estimator. Logistic regression uses log odds; tree probability output and XGBoost log margins are distinct. The saved conversion and calibration mapping produce final probability. XGBoost margins require an expit transformation before its probability-based calibrator.

In [ ]:
for path in [OUT,OUT/'customer_model']:
    display(pd.read_csv(path/'feature_importance.csv').head(15))
    print(json.loads((path/'explainability.json').read_text()))
display(Image(filename=str(OUT/'figures/feature_importance.png')))